In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [2]:
with_bias_file = r"D:\PROJECTS\YAANENDRIYA\data\ALiMU_working\synthetic_imu_data_01_withBias_1h_stat_low_high.txt"

without_bias_file = r"D:\PROJECTS\YAANENDRIYA\data\ALiMU_working\synthetic_imu_data_02_no_bias_stat_low_high.txt"

df_bias = pd.read_csv(with_bias_file)
df_no_bias = pd.read_csv(without_bias_file)

print("With bias:", df_bias.shape)
print("No bias  :", df_no_bias.shape)

With bias: (180000, 18)
No bias  : (180000, 18)


In [3]:
for axis in ["x", "y", "z"]:
    df_bias[f"error_{axis}"] = (
        df_bias[f"IMU_low_a{axis}"] -
        df_bias[f"IMU_high_a{axis}"]
    )

    df_no_bias[f"error_{axis}"] = (
        df_no_bias[f"IMU_low_a{axis}"] -
        df_no_bias[f"IMU_high_a{axis}"]
    )

target_cols = ["error_x", "error_y", "error_z"]

print("WITH BIAS")
print(df_bias[target_cols].describe())

print("\nWITHOUT BIAS")
print(df_no_bias[target_cols].describe())

WITH BIAS
             error_x        error_y        error_z
count  180000.000000  180000.000000  180000.000000
mean       -1.867105       2.227332       0.886849
std         0.053651       0.051642       0.083363
min        -2.058673       2.083275       0.692500
25%        -1.888061       2.187679       0.826400
50%        -1.857460       2.220315       0.860200
75%        -1.832643       2.270846       0.946000
max        -1.715456       2.385355       1.137100

WITHOUT BIAS
             error_x        error_y        error_z
count  180000.000000  180000.000000  180000.000000
mean        0.208155       0.134104       0.021113
std         0.040205       0.038098       0.044255
min         0.067464       0.001602      -0.125400
25%         0.183441       0.103999      -0.011700
50%         0.213093       0.136123       0.028300
75%         0.236528       0.162944       0.055100
max         0.340301       0.259578       0.151500


In [4]:
feature_cols = [
    "IMU_low_ax",
    "IMU_low_ay",
    "IMU_low_az"
]

def train_test_ridge(df):

    X = df[feature_cols].values
    Y = df[target_cols].values

    split = int(0.8 * len(df))

    X_train = X[:split]
    X_test = X[split:]

    Y_train = Y[:split]
    Y_test = Y[split:]

    scaler = StandardScaler()

    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    model = Ridge(alpha=1.0)
    model.fit(X_train_s, Y_train)

    prediction = model.predict(X_test_s)

    return Y_train, Y_test, prediction

Yb_train, Yb_test, pred_b = train_test_ridge(df_bias)
Yn_train, Yn_test, pred_n = train_test_ridge(df_no_bias)

In [5]:
for i, axis in enumerate(["X", "Y", "Z"]):

    # WITH BIAS
    baseline_b = np.full(
        len(Yb_test),
        Yb_train[:, i].mean()
    )

    rmse_b_base = np.sqrt(
        mean_squared_error(Yb_test[:, i], baseline_b)
    )

    rmse_b_ml = np.sqrt(
        mean_squared_error(Yb_test[:, i], pred_b[:, i])
    )

    mae_b_ml = mean_absolute_error(
        Yb_test[:, i],
        pred_b[:, i]
    )

    imp_b = (1 - rmse_b_ml / rmse_b_base) * 100

    # WITHOUT BIAS
    baseline_n = np.full(
        len(Yn_test),
        Yn_train[:, i].mean()
    )

    rmse_n_base = np.sqrt(
        mean_squared_error(Yn_test[:, i], baseline_n)
    )

    rmse_n_ml = np.sqrt(
        mean_squared_error(Yn_test[:, i], pred_n[:, i])
    )

    mae_n_ml = mean_absolute_error(
        Yn_test[:, i],
        pred_n[:, i]
    )

    imp_n = (1 - rmse_n_ml / rmse_n_base) * 100

    print(f"\n{axis}-axis")

    print("WITH BIAS")
    print(f"Baseline RMSE : {rmse_b_base:.6f}")
    print(f"ML RMSE       : {rmse_b_ml:.6f}")
    print(f"ML MAE        : {mae_b_ml:.6f}")
    print(f"Improvement   : {imp_b:.2f}%")

    print("WITHOUT BIAS")
    print(f"Baseline RMSE : {rmse_n_base:.6f}")
    print(f"ML RMSE       : {rmse_n_ml:.6f}")
    print(f"ML MAE        : {mae_n_ml:.6f}")
    print(f"Improvement   : {imp_n:.2f}%")


X-axis
WITH BIAS
Baseline RMSE : 0.107910
ML RMSE       : 0.002783
ML MAE        : 0.002223
Improvement   : 97.42%
WITHOUT BIAS
Baseline RMSE : 0.028747
ML RMSE       : 0.002769
ML MAE        : 0.002209
Improvement   : 90.37%

Y-axis
WITH BIAS
Baseline RMSE : 0.057109
ML RMSE       : 0.002774
ML MAE        : 0.002206
Improvement   : 95.14%
WITHOUT BIAS
Baseline RMSE : 0.045540
ML RMSE       : 0.002775
ML MAE        : 0.002213
Improvement   : 93.91%

Z-axis
WITH BIAS
Baseline RMSE : 0.105301
ML RMSE       : 0.002798
ML MAE        : 0.002234
Improvement   : 97.34%
WITHOUT BIAS
Baseline RMSE : 0.078043
ML RMSE       : 0.002781
ML MAE        : 0.002215
Improvement   : 96.44%


In [6]:
for name, df in [
    ("WITH BIAS", df_bias),
    ("WITHOUT BIAS", df_no_bias)
]:
    print("\n", name)

    for axis in ["x", "y", "z"]:
        rolling_mean = df[f"error_{axis}"].rolling(10000).mean()

        print(
            axis.upper(),
            "rolling mean range:",
            round(rolling_mean.min(), 6),
            "to",
            round(rolling_mean.max(), 6)
        )


 WITH BIAS
X rolling mean range: -1.99304 to -1.7876
Y rolling mean range: 2.155618 to 2.310535
Z rolling mean range: 0.779354 to 1.054556

 WITHOUT BIAS
X rolling mean range: 0.127055 to 0.270737
Y rolling mean range: 0.082499 to 0.178892
Z rolling mean range: -0.061573 to 0.07852
